<a href="https://colab.research.google.com/github/Kishoby/Conceptual-Research_Hybrid-Approach/blob/Multi-Target/Multi_Target_without_Iterations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =========================================================
# TRADITIONAL ML - MULTI TARGET
# ONLY 2 TABLES:
#   1. Training Results
#   2. Testing Results
# =========================================================

import pandas as pd
import numpy as np

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.multioutput import MultiOutputRegressor

# ---------------------------------------------------------
# 1. LOAD CLEANED MULTI-TARGET DATASET
# ---------------------------------------------------------
multi_target_df = pd.read_csv("/content/drive/MyDrive/Research v2/Research 18.03.2026/Multi Target/cleaned_multi_target_dataset (1).csv")

print("Dataset shape:", multi_target_df.shape)
display(multi_target_df.head())

# ---------------------------------------------------------
# 2. DEFINE TARGETS AND FEATURES
# ---------------------------------------------------------
targets = [
    "air_quality_PM2.5",
    "temperature_celsius",
    "humidity"
]

features = [col for col in multi_target_df.columns if col not in targets]

print("Targets:", targets)
print("Number of features:", len(features))

# ---------------------------------------------------------
# 3. TRAIN / TEST SPLIT
# ---------------------------------------------------------
split_index = int(0.8 * len(multi_target_df))

train_df = multi_target_df.iloc[:split_index].copy()
test_df = multi_target_df.iloc[split_index:].copy()

X_train = train_df[features]
y_train = train_df[targets]

X_test = test_df[features]
y_test = test_df[targets]

print("Train shape:", X_train.shape, y_train.shape)
print("Test shape :", X_test.shape, y_test.shape)

# ---------------------------------------------------------
# 4. METRIC FUNCTION
# ---------------------------------------------------------
def calculate_metrics_per_target(y_true, y_pred, model_name, dataset_type, target_names):
    rows = []

    for i, target in enumerate(target_names):
        mse = mean_squared_error(y_true.iloc[:, i], y_pred[:, i])
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(y_true.iloc[:, i], y_pred[:, i])
        r2 = r2_score(y_true.iloc[:, i], y_pred[:, i])
        acc = r2 * 100

        rows.append([
            dataset_type,
            model_name,
            target,
            mse,
            rmse,
            mae,
            r2,
            acc
        ])

    return pd.DataFrame(
        rows,
        columns=["Dataset", "Model", "Target", "MSE", "RMSE", "MAE", "R2", "Accuracy (%)"]
    ).round(3)

# ---------------------------------------------------------
# 5. RESULT STORAGE
# ---------------------------------------------------------
training_results = []
testing_results = []

# =========================================================
# MODEL 1 — SGD REGRESSOR
# =========================================================
from sklearn.linear_model import SGDRegressor

scaler_sgd = StandardScaler()

X_train_sgd = scaler_sgd.fit_transform(X_train)
X_test_sgd = scaler_sgd.transform(X_test)

sgd_model = MultiOutputRegressor(SGDRegressor(random_state=42))
sgd_model.fit(X_train_sgd, y_train)

sgd_train_pred = sgd_model.predict(X_train_sgd)
sgd_test_pred = sgd_model.predict(X_test_sgd)

training_results.append(
    calculate_metrics_per_target(y_train, sgd_train_pred, "SGD Regressor", "Training", targets)
)
testing_results.append(
    calculate_metrics_per_target(y_test, sgd_test_pred, "SGD Regressor", "Testing", targets)
)

# =========================================================
# MODEL 2 — RANDOM FOREST
# =========================================================
from sklearn.ensemble import RandomForestRegressor

rf_model = MultiOutputRegressor(
    RandomForestRegressor(
        n_estimators=30,
        max_depth=10,
        n_jobs=-1,
        random_state=42
    )
)

rf_model.fit(X_train, y_train)

rf_train_pred = rf_model.predict(X_train)
rf_test_pred = rf_model.predict(X_test)

training_results.append(
    calculate_metrics_per_target(y_train, rf_train_pred, "Random Forest", "Training", targets)
)
testing_results.append(
    calculate_metrics_per_target(y_test, rf_test_pred, "Random Forest", "Testing", targets)
)

# =========================================================
# MODEL 3 — XGBOOST
# =========================================================
import xgboost as xgb

xgb_train_preds = []
xgb_test_preds = []

for i in range(len(targets)):
    model = xgb.XGBRegressor(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42
    )

    model.fit(X_train, y_train.iloc[:, i])

    xgb_train_preds.append(model.predict(X_train))
    xgb_test_preds.append(model.predict(X_test))

xgb_train_pred = np.column_stack(xgb_train_preds)
xgb_test_pred = np.column_stack(xgb_test_preds)

training_results.append(
    calculate_metrics_per_target(y_train, xgb_train_pred, "XGBoost", "Training", targets)
)
testing_results.append(
    calculate_metrics_per_target(y_test, xgb_test_pred, "XGBoost", "Testing", targets)
)

# =========================================================
# MODEL 4 — LIGHTGBM
# =========================================================
import lightgbm as lgb

lgb_train_preds = []
lgb_test_preds = []

params = {
    "objective": "regression",
    "metric": "rmse",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "verbose": -1
}

for i in range(len(targets)):
    train_data = lgb.Dataset(X_train, label=y_train.iloc[:, i])
    model = lgb.train(params, train_data, num_boost_round=100)

    lgb_train_preds.append(model.predict(X_train))
    lgb_test_preds.append(model.predict(X_test))

lgb_train_pred = np.column_stack(lgb_train_preds)
lgb_test_pred = np.column_stack(lgb_test_preds)

training_results.append(
    calculate_metrics_per_target(y_train, lgb_train_pred, "LightGBM", "Training", targets)
)
testing_results.append(
    calculate_metrics_per_target(y_test, lgb_test_pred, "LightGBM", "Testing", targets)
)

# =========================================================
# MODEL 5 — CNN
# =========================================================
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, Flatten, Dense, Input

scaler_cnn = StandardScaler()

X_train_cnn = scaler_cnn.fit_transform(X_train)
X_test_cnn = scaler_cnn.transform(X_test)

X_train_cnn = X_train_cnn.reshape((X_train_cnn.shape[0], X_train_cnn.shape[1], 1))
X_test_cnn = X_test_cnn.reshape((X_test_cnn.shape[0], X_test_cnn.shape[1], 1))

cnn_model = Sequential([
    Input(shape=(X_train_cnn.shape[1], 1)),
    Conv1D(filters=32, kernel_size=2, activation="relu", padding="same"),
    Flatten(),
    Dense(64, activation="relu"),
    Dense(len(targets))
])

cnn_model.compile(optimizer="adam", loss="mse")

cnn_model.fit(X_train_cnn, y_train, epochs=10, batch_size=32, verbose=0)

cnn_train_pred = cnn_model.predict(X_train_cnn, verbose=0)
cnn_test_pred = cnn_model.predict(X_test_cnn, verbose=0)

training_results.append(
    calculate_metrics_per_target(y_train, cnn_train_pred, "CNN", "Training", targets)
)
testing_results.append(
    calculate_metrics_per_target(y_test, cnn_test_pred, "CNN", "Testing", targets)
)

# =========================================================
# 6. FINAL 2 TABLES ONLY
# =========================================================
training_table = pd.concat(training_results, ignore_index=True)
testing_table = pd.concat(testing_results, ignore_index=True)

print("TRAINING RESULTS TABLE")
display(training_table)

print("TESTING RESULTS TABLE")
display(testing_table)

# =========================================================
# 7. DOWNLOAD ONLY 2 TABLES
# =========================================================
from google.colab import files

training_table.to_csv("multi_target_training_results.csv", index=False)
testing_table.to_csv("multi_target_testing_results.csv", index=False)

files.download("multi_target_training_results.csv")
files.download("multi_target_testing_results.csv")

Dataset shape: (130003, 18)


,visibility_km,last_updated_epoch,air_quality_Ozone,latitude,air_quality_PM10,condition_text,air_quality_Sulphur_dioxide,uv_index,air_quality_Nitrogen_dioxide,feels_like_celsius,precip_mm,cloud,pressure_mb,longitude,air_quality_Carbon_Monoxide,air_quality_PM2.5,temperature_celsius,humidity
0,16.0,1715849100,62.2,46.60,7.1,2,0.2,1.0,2.5,16.1,0.00,0,1012.0,-120.49,198.6,6.3,16.1,58
1,10.0,1715849100,23.3,14.10,25.3,32,1.4,1.0,3.7,25.3,0.28,37,1017.0,-87.22,377.2,19.0,23.0,78
2,10.0,1715849100,5.9,13.71,28.1,23,7.5,1.0,7.7,30.2,0.30,50,1010.0,-89.20,460.6,20.4,26.0,94
3,5.0,1715849100,0.4,14.62,178.1,19,19.3,1.0,35.0,20.0,0.09,100,1019.0,-90.53,2243.0,132.0,20.0,88
4,10.0,1715849100,34.0,17.25,32.1,30,0.2,1.0,0.3,29.6,0.00,94,1007.0,-88.77,307.1,7.7,26.0,89


Targets: ['air_quality_PM2.5', 'temperature_celsius', 'humidity']
Number of features: 15
Train shape: (104002, 15) (104002, 3)
Test shape : (26001, 15) (26001, 3)
TRAINING RESULTS TABLE


,Dataset,Model,Target,MSE,RMSE,MAE,R2,Accuracy (%)
0,Training,SGD Regressor,air_quality_PM2.5,3293.756,57.391,17.625,-1.068,-106.786
1,Training,SGD Regressor,temperature_celsius,3.216,1.793,1.211,0.959,95.893
2,Training,SGD Regressor,humidity,231.543,15.217,11.892,0.603,60.306
3,Training,Random Forest,air_quality_PM2.5,36.154,6.013,3.080,0.977,97.730
4,Training,Random Forest,temperature_celsius,0.633,0.796,0.520,0.992,99.192
5,Training,Random Forest,humidity,105.595,10.276,7.831,0.819,81.897
6,Training,XGBoost,air_quality_PM2.5,56.561,7.521,3.495,0.964,96.449
7,Training,XGBoost,temperature_celsius,0.724,0.851,0.585,0.991,99.075
8,Training,XGBoost,humidity,101.853,10.092,7.720,0.825,82.539
9,Training,LightGBM,air_quality_PM2.5,55.607,7.457,3.467,0.965,96.509


TESTING RESULTS TABLE


,Dataset,Model,Target,MSE,RMSE,MAE,R2,Accuracy (%)
0,Testing,SGD Regressor,air_quality_PM2.5,586.614,24.220,12.855,0.020,2.019
1,Testing,SGD Regressor,temperature_celsius,2.287,1.512,1.119,0.982,98.158
2,Testing,SGD Regressor,humidity,278.936,16.701,12.558,0.418,41.814
3,Testing,Random Forest,air_quality_PM2.5,47.631,6.901,3.331,0.920,92.044
4,Testing,Random Forest,temperature_celsius,0.934,0.966,0.617,0.992,99.248
5,Testing,Random Forest,humidity,201.045,14.179,10.184,0.581,58.062
6,Testing,XGBoost,air_quality_PM2.5,66.408,8.149,3.525,0.889,88.908
7,Testing,XGBoost,temperature_celsius,3.059,1.749,0.930,0.975,97.537
8,Testing,XGBoost,humidity,189.904,13.781,9.869,0.604,60.386
9,Testing,LightGBM,air_quality_PM2.5,73.300,8.562,3.491,0.878,87.757


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>